In [39]:
import duckdb
import pandas as pd
import os

cwd = os.getcwd()
dir = rf"./datsets/news"
con = duckdb.connect("research.duckdb")

In [43]:
con.execute("SHOW TABLES").df()

,name
0,developments
1,eps_features
2,eps_raw
3,extended_universe
4,historical_ciq_formula
5,universe


In [ ]:
def merge_news_csv():
    # read and clean the news spreadsheets
    # add the cleaned csv's to a single database
    files = [file for file in os.listdir(dir)]

    for i, file in enumerate(files):
        filepath = os.path.join(dir, file)
        df = pd.read_csv(filepath, encoding='latin1')
        df1 = df.iloc[4:, :]
        cols = ["dev_date", "dev_id", "dev_type", "headline", "description", "Entity ID"]
        df1.columns = cols
        df1['IQIDs'] = df1['Entity ID'].str.findall(r'(\d+)')
        df1 = df1[["dev_date", "IQIDs", "dev_id", "dev_type", "headline", "description"]]


        if i == 0:
            # First file: Create the table from the dataframe
            con.execute("CREATE OR REPLACE TABLE developments AS SELECT * FROM df1")
        else:
            # Subsequent files: Append the dataframe to the existing table
            con.execute("INSERT INTO developments SELECT * FROM df1")
        
        print(f"Processed {file}")


In [45]:
con.execute("SELECT * FROM developments LIMIT 10").df()

,dev_date,IQIDs,dev_id,dev_type,headline,description
0,24/03/2015,[4587650],10292406,Product-related Announcement,The ING Group Transforms HR in the Cloud with ...,ServiceNow announced that the ING Group is suc...
1,09/07/2015,[4004426],17232373,Buyback: Tranche Update,Tranche Update on V.F. Corporation (NYSE:VFC)'...,"From April 5, 2015 to July 4, 2015, the compan..."
2,23/02/2015,[4040505],10292436,Client Announcement,General Dynamics Lands $415 Million Contract t...,General Dynamics Corp. (GD) announced it has w...
3,10/08/2014,[4344006],10045646,Client Announcement,Immersion Corporation Announces License with S...,Immersion Corporation announced that it has ex...
4,13/10/2014,"[4839198, 114523, 4066414, 4067461, 110587]",18037181,M&A: Transaction Announcement,"Goldman Sachs Asset Management, L.P., Merchant...","Goldman Sachs Asset Management, L.P., Merchant..."
5,13/11/2015,[4071818],15090194,Sales or Trading Statement Call,"W.W. Grainger, Inc. - Sales/ Trading Statement...","W.W. Grainger, Inc. - Sales/ Trading Statement..."
6,18/05/2015,"[4509697, 4996839]",10371901,Client Announcement,Midlands Highway Alliance Awards 30-Million C...,The Midlands Highway Alliance (MHA) has awarde...
7,17/03/2015,[103336],10322702,Other Executive or Board Change,W. R. Berkley Corporation Appoints Dana R. Fra...,W. R. Berkley Corporation announced the appoin...
8,14/09/2015,[4041896],18708408,Fixed Income Offering,Citigroup Inc. has announced a Fixed-Income Of...,Citigroup Inc. has announced a Fixed-Income Of...
9,13/10/2015,[5257077],10581942,Product-related Announcement,Appboy Launches Intelligence Suite and Brings ...,Appboy launched several new products that help...
